# Data Extraction (DATA_ONLY Mode) — Speculative Decoding Training

This notebook demonstrates how to extract hidden states from a verifier model using the
`SpeculativeDecodingTrainer` in **DATA_ONLY** mode on Red Hat OpenShift AI.

## What Does DATA_ONLY Mode Do?

DATA_ONLY mode extracts hidden states from the verifier model without performing any training.
The SDK deploys a managed vLLM sidecar alongside the job pod to serve the verifier model.
The sidecar processes the dataset and writes hidden state tensors (`.safetensors` files) to
the output PVC. This is the first step of a two-step workflow — extract once, then experiment
with training hyperparameters many times using [TRAIN_ONLY](../train-only/) mode.

## Speculative Decoding Overview

Large language models generate tokens one at a time, and each token requires reading the entire
model from GPU memory — making inference **memory-bound**. Speculative decoding exploits this:
a small, fast **draft model** (~1 GB) guesses the next several tokens, then the large **verifier
model** checks all guesses in a single forward pass. The output is mathematically identical to
normal decoding — no quality loss.

**Eagle3** is a draft model architecture that reads hidden states from four intermediate layers
of the verifier (not just the final logits), giving it richer context for more accurate
predictions.

## Dataset

This example uses the `ultrachat` built-in dataset (multi-turn conversational data).

## Hardware Requirements

| Component | GPU | CPU | Memory | Notes |
|-----------|-----|-----|--------|-------|
| vLLM sidecar | 1× NVIDIA L40S / A100 | 4 cores | 96Gi | Serves verifier for hidden state extraction |

> **Note:** No training container is deployed in DATA_ONLY mode — only the vLLM sidecar runs.

## Setup

Install the Kubeflow SDK and import required dependencies.

In [ ]:
# Install the Kubeflow SDK from the OpenDataHub fork (includes SpeculativeDecodingTrainer)
!pip install --no-cache-dir --force-reinstall --no-deps git+https://github.com/opendatahub-io/kubeflow-sdk.git@main

# Structured logging library (dependency for SDK progress tracking)
!pip install structlog

# Kubeflow Trainer API — provides TrainerClient, KubernetesBackendConfig, and job management
!pip install --no-cache-dir --force-reinstall --index-url https://pypi.org/simple kubeflow-trainer-api==2.3.0

In [ ]:
import os

import kubeflow

# Backend config tells the TrainerClient how to connect to the cluster
from kubeflow.common.types import KubernetesBackendConfig

# TrainerClient is the main entry point for submitting, monitoring, and deleting TrainJobs
from kubeflow.trainer import TrainerClient

# Name option lets you assign an explicit name to a TrainJob (otherwise auto-generated)
from kubeflow.trainer.options.common import Name

# Red Hat OpenShift AI extensions for speculative decoding training
from kubeflow.trainer.rhai import (
    SpeculativeDecodingTrainer,  # High-level trainer that wraps all four modes
    SpeculatorConfig,  # Fine-grained config: layer IDs, architecture, scheduler, etc.
    SpeculatorMode,  # Enum: DATA_ONLY, TRAIN_ONLY, OFFLINE, ONLINE
    SpeculatorType,  # Enum: EAGLE3 (currently the only supported type)
)

# Kubernetes Python client — used to configure API server auth and create the API client
from kubernetes import client

print(f"Kubeflow SDK version: {kubeflow.__version__}")
print("All imports successful")

In [ ]:
# Verify the SDK loaded correctly by inspecting the available enum values and defaults.
# This confirms that SpeculatorMode, SpeculatorType, and SpeculatorConfig are importable
# and behave as expected before proceeding to cluster authentication.
print(f"Modes: {[m.value for m in SpeculatorMode]}")
print(f"Types: {[t.value for t in SpeculatorType]}")
print(f"Config defaults: {SpeculatorConfig()}")
print("SDK ready")

## Authenticate to your OpenShift Cluster

Provide your OpenShift API server URL, authentication token, and HuggingFace token.
Update `PVC_NAME` to match the name of your shared RWX PersistentVolumeClaim.

In [ ]:
# ============================================================================
# CLUSTER AUTHENTICATION
# ============================================================================
# Replace these with your OpenShift cluster API server URL and bearer token.
# In OpenShift AI workbenches, these may be available as environment variables
# (OPENSHIFT_API_URL, NOTEBOOK_USER_TOKEN) — but for clarity we set them explicitly.
api_server = "<REPLACE WITH OPENSHIFT SERVER>"
token = "<REPLACE WITH OPENSHIFT TOKEN>"

# HuggingFace token — required for gated models; recommended for all models to avoid rate limits
HF_TOKEN = "<REPLACE WITH HF TOKEN>"

# ============================================================================
# KUBERNETES CLIENT CONFIGURATION
# ============================================================================
# The Configuration object holds the API server URL, auth token, and TLS settings.
# This is passed to the ApiClient, which the TrainerClient uses for all cluster operations.
configuration = client.Configuration()
configuration.host = api_server

# Uncomment if your cluster API server uses a self-signed certificate or an untrusted CA
# configuration.verify_ssl = False

configuration.api_key = {"authorization": f"Bearer {token}"}
api_client = client.ApiClient(configuration)

# ============================================================================
# PVC CONFIGURATION
# ============================================================================
# PVC_NAME must match the ReadWriteMany (RWX) PVC attached to your workbench.
# The notebook sees it at /opt/app-root/src/<pvc-name> (OpenShift AI convention).
# Training pods see it at /mnt/kubeflow-checkpoints (SDK constant CHECKPOINT_MOUNT_PATH).
PVC_NAME = "shared"
NOTEBOOK_SHARED_PATH = f"/opt/app-root/src/{PVC_NAME}"
SDK_MOUNT_PATH = "/mnt/kubeflow-checkpoints"

# Quick sanity check to help users discover the right workbench mount
if not os.path.exists(NOTEBOOK_SHARED_PATH):
    print(
        f"Warning: Expected workbench PVC mount not found at: {NOTEBOOK_SHARED_PATH}\n"
        "If your PVC has a different name or mount, update PVC_NAME above.\n"
        "Tip: in a workbench, PVCs are typically under /opt/app-root/src/."
    )

# ============================================================================
# TRAINER CLIENT
# ============================================================================
# Create the TrainerClient — the main SDK entry point for submitting and managing TrainJobs.
trainer_client = TrainerClient(
    backend_config=KubernetesBackendConfig(
        client_configuration=api_client.configuration
    )
)

# ClusterTrainingRuntime (CTR) for DATA_ONLY mode.
# This CTR deploys a vLLM sidecar for hidden state extraction.
# Must be pre-installed on the cluster by an admin.
DATA_EXTRACT_CTR = "vllm-extract-cuda"

# Verify the CTR exists on the cluster
available_runtimes = {r.name for r in trainer_client.list_runtimes()}
status = (
    "Found"
    if DATA_EXTRACT_CTR in available_runtimes
    else "WARNING: not found on cluster"
)
print(f"CTR '{DATA_EXTRACT_CTR}': {status}")

print(f"\nAPI Server: {api_server}")
print(f"PVC name: {PVC_NAME}")
print(f"Workbench PVC mount: {NOTEBOOK_SHARED_PATH}")
print(f"Training pod PVC mount (SDK): {SDK_MOUNT_PATH}")

## Configuration

The following constants configure the data extraction run. The verifier model is
[Qwen/Qwen3-8B](https://huggingface.co/Qwen/Qwen3-8B), a 36-layer transformer,
specified by its HuggingFace model ID. The training pods download the model
automatically — no manual pre-download is required (`HF_TOKEN` provides
authentication).

All output paths use **PVC URIs** (`pvc://<pvc-name>/<path>`), which the SDK
resolves to container mount paths internally.

`RUN_NAME` creates a namespace on the PVC for each experiment — change it to start
a fresh run without overwriting previous results.

In [ ]:
# Unique run identifier — namespaces all output paths on the PVC.
# Change this to start a fresh experiment without overwriting previous results.
RUN_NAME = "run-01"

# Set the Verifier Model for the training job.
VERIFIER_MODEL = "Qwen/Qwen3-8B"

# Eagle3 reads hidden states from 4 intermediate layers of the verifier.
# Qwen3-8B has 36 transformer layers (indexed 1-36).
# Layers chosen: early (3), mid (18), late (33), and final (36) — giving the
# draft model a spread of low-level, mid-level, and high-level representations.
TARGET_LAYER_IDS = [3, 18, 33, 36]

# Resources for the vLLM sidecar that serves the verifier during extraction.
# 1 GPU is sufficient since vLLM only runs inference (no training).
# 96Gi memory is needed because vLLM loads the full model weights into CPU/GPU memory.
VLLM_RESOURCES = {
    "nvidia.com/gpu": 1,
    "cpu": "4",
    "memory": "96Gi",
}

# Data extraction parameters
TOTAL_SEQ_LEN = 2048  # Maximum sequence length for extraction
MAX_SAMPLES = 500  # Cap on the number of dataset samples to process

print("DATA_ONLY Configuration:")
print(f"  Run name:          {RUN_NAME}")
print(f"  Verifier model:    {VERIFIER_MODEL}")
print(f"  Target layers:     {TARGET_LAYER_IDS}")
print(f"  vLLM GPUs:         {VLLM_RESOURCES['nvidia.com/gpu']}")
print(f"  Sequence length:   {TOTAL_SEQ_LEN}")
print(f"  Max samples:       {MAX_SAMPLES}")

## Data Extraction (DATA_ONLY Mode)

The `DATA_ONLY` mode extracts hidden states from the verifier model without performing
any training. The SDK deploys a managed vLLM sidecar alongside the job pod to serve the
verifier model. The sidecar processes the dataset and writes hidden state tensors
(`.safetensors` files) to the output PVC.

This is useful when you want to:
- Separate data extraction from training (extract once, experiment many times)
- Share extracted data across multiple training runs with different hyperparameters

**Key parameters explained:**
- `speculator_type` — Explicitly selects the Eagle3 architecture for the draft model
- `vllm_gpu_memory_utilization` — Fraction of GPU memory the vLLM sidecar can use (0.9 = 90%)
- `regenerate_responses` — When `True`, the vLLM sidecar generates new responses from the
  dataset prompts before extracting hidden states, rather than using the dataset's existing responses
- `datagen_concurrency` — Number of concurrent data generation workers
- `hidden_states_dtype` — Data type for saved hidden state tensors (`bfloat16` saves disk space)

**Not needed**: `training_resources` (no training happens in this mode)

We use the `ultrachat` built-in dataset for this example.

In [ ]:
# Job name follows a consistent naming convention: <architecture>-<mode>-<run>
DATA_JOB = f"eagle3-data-{RUN_NAME}"

# Output directory on PVC — hidden states will be written to <output_dir>/hidden_states/
DATA_ONLY_OUTPUT = f"pvc://{PVC_NAME}/speculator/{RUN_NAME}"

# Configure the DATA_ONLY trainer.
# This mode ONLY extracts hidden states — no training happens.
# The SDK deploys a managed vLLM sidecar to serve the verifier model.
data_only_trainer = SpeculativeDecodingTrainer(
    mode=SpeculatorMode.DATA_ONLY,  # Extract hidden states only, no training
    speculator_type=SpeculatorType.EAGLE3,  # Use Eagle3 draft model architecture
    verifier_model=VERIFIER_MODEL,  # HuggingFace model ID — downloaded automatically by the job
    dataset_name="ultrachat",  # Built-in multi-turn conversational dataset
    max_samples=MAX_SAMPLES,  # Cap on the number of dataset samples to process
    total_seq_len=TOTAL_SEQ_LEN,  # Truncate/pad sequences to 2048 tokens
    vllm_resources=VLLM_RESOURCES,  # GPU/CPU/memory for the vLLM sidecar
    vllm_gpu_memory_utilization=0.9,  # Let vLLM use 90% of GPU memory for KV cache
    regenerate_responses=True,  # Generate fresh responses from prompts (not reuse dataset answers)
    enable_progression_tracking=True,  # Enable SDK-side progress polling
    packages_to_install=["speculators==0.6.0", "torchvision==0.24.0"],
    output_dir=DATA_ONLY_OUTPUT,  # PVC path where hidden states are saved
    config=SpeculatorConfig(
        target_layer_ids=TARGET_LAYER_IDS,  # Which verifier layers to extract hidden states from
        datagen_concurrency=4,  # Number of parallel data generation workers
        hidden_states_dtype="bfloat16",  # Save tensors in bfloat16 to halve disk usage
    ),
    env={"HF_TOKEN": HF_TOKEN},  # Pass HuggingFace token to pods for gated model access
)

print("DATA_ONLY Configuration:")
print(f"  Job name:      {DATA_JOB}")
print(f"  Mode:          {data_only_trainer.mode.value}")
print(f"  Verifier:      {data_only_trainer.verifier_model}")
print(f"  Dataset:       {data_only_trainer.dataset_name}")
print(f"  Max samples:   {data_only_trainer.max_samples}")
print(f"  Target layers: {data_only_trainer.config.target_layer_ids}")
print(f"  Output dir:    {data_only_trainer.output_dir}")

In [ ]:
# Submit the DATA_ONLY TrainJob to the cluster.
# Name(...) assigns an explicit job name (otherwise the SDK auto-generates one).
# The runtime selects the CTR that includes a vLLM sidecar for hidden state extraction.
trainer_client.train(
    options=[Name(name=DATA_JOB)],
    trainer=data_only_trainer,
    runtime=DATA_EXTRACT_CTR,
)

print(f"DATA_ONLY job submitted: {DATA_JOB}")
print("\nMonitor logs with:")
print(f"  oc logs -f -l batch.kubernetes.io/job-name={DATA_JOB}-node-0 -c node")

In [ ]:
# Check the current status of the DATA_ONLY job (Pending, Running, Succeeded, Failed).
# Re-run this cell periodically to poll for completion.
trainer_client.get_job(DATA_JOB)

## Cleanup

Delete the TrainJob when you are done. Uncomment the line below to delete.

In [ ]:
# Delete the completed TrainJob to free cluster resources (pods, volumes, etc.).
# Note: Deleting a job does NOT delete the output data on the PVC —
# hidden states remain available for the TRAIN_ONLY step.

# trainer_client.delete_job(DATA_JOB)

# print("TrainJob deleted.")

## Summary

This notebook extracted hidden states from Qwen3-8B using a managed vLLM sidecar
and the `ultrachat` dataset. The extracted data is stored on the PVC at
`pvc://<pvc-name>/speculator/<run-name>/hidden_states/` and can be reused across
multiple training runs.

### Next Steps

- **Train the draft model**: Run the [TRAIN_ONLY](../train-only/) notebook to train
  an Eagle3 draft model from the hidden states extracted in this step. You can iterate
  on training hyperparameters (`epochs`, `lr`, `num_layers`, etc.) without re-running
  the expensive extraction step.

### Alternative Modes

- [OFFLINE](../offline/) — Extract + train in a single job using an external vLLM endpoint
- [ONLINE](../online/) — Fully managed end-to-end extraction and training in one step